# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR² dataset (Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution) using the `mlcroissant` library.

### Dataset Source
The dataset is defined via a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema describes record sets and their fields. Here, we enumerate all record sets, their `@id` values (unique identifiers), and the contained fields/columns. Using `mlcroissant`, you can access the schema objects programmatically.

In [ ]:
# Enumerate record sets and their fields using their @id
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"RecordSet: {rs['@id']} - {rs.get('name', '[no name]')}")
    print("  Fields:")
    for field in rs["fields"]:
        print(f"    Field: {field['@id']} ({field.get('name', '[no name]')})")
    print()

# Print an example record from one record set
if record_sets:
    rs_id = record_sets[0]["@id"]
    for x in dataset.records(record_set=rs_id):
        print(f"Sample record from RecordSet {rs_id}:\n{x}")
        break

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis.

Each entity is referenced by its `@id` as per schema best practices. Here, we extract each record set, show their columns, and display the head.

In [ ]:
# Collect RecordSet IDs
record_set_ids = [rs["@id"] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns in RecordSet {record_set_id}:\n{df.columns.tolist()}")
    print(f"Head of RecordSet {record_set_id}:")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, and grouping data.

All variables, fields, and columns are referenced via their schema `@id` for consistency. Here, we select the main record set and perform EDA on a numeric field, grouping by a key attribute.

In [ ]:
# Select the primary record set
primary_rs_id = record_set_ids[0]
df = dataframes[primary_rs_id]

# Display available columns (@id derived names)
print(f"Available columns for EDA in {primary_rs_id}:")
print(df.columns.tolist())

# Choose a numeric field (e.g. 'age' or similar field @id)
# For demonstration, let's look for a field containing 'age' in its @id or column list
numeric_field_id = None
for col in df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Use a fallback numeric field
    numeric_cols = df.select_dtypes(include='number').columns
    if len(numeric_cols):
        numeric_field_id = numeric_cols[0]

print(f"Selected numeric field for filtering and normalization: {numeric_field_id}")

# Apply filtering if field exists
threshold = 50  # Example, age > 50
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, normalized_col]].head())
    
    # Group by a categorical field, e.g. with 'sex', 'msi', 'anatomy' in @id/column name
    group_field_id = None
    for col in df.columns:
        if any(key in col.lower() for key in ['sex', 'msi', 'anatomy']):
            group_field_id = col
            break
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between key fields.

Here, we plot the distribution of the selected numeric field and show field relationships by group.

In [ ]:
# Visualizations
import matplotlib.pyplot as plt

if numeric_field_id in df.columns:
    plt.figure(figsize=(8,5))
    df[numeric_field_id].hist(bins=15)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    # Boxplot by group_field_id if available
    if 'group_field_id' in locals() and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        df.boxplot(column=[numeric_field_id], by=group_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle('')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded and explored the FAIR² clinical colorectal cancer dataset using the Croissant schema and mlcroissant library.
- All entities (record sets, fields, columns) were referenced via their `@id` values for clarity and reproducibility.
- Data extraction and filtering highlighted demographic distributions (e.g. age) and clinical grouping (e.g. MSI status, anatomical location).
- Visualizations revealed distributions and allow segmentation by biomarker or anatomical category.
- The dataset supports clinical hypothesis generation and predictive modeling for second primary CRC in survivors.